# De-Inflated Fair-Gated SAE Unlearning-Edit Test — Demo

This notebook is a **light-weight, CPU-friendly demo** of the analysis behind the experiment
**"De-Inflated Fair-Gated SAE Unlearning-Edit Test on Concentrated Absorbers"** (M1''').

The full experiment runs `google/gemma-2-2b` + a Gemma-Scope `layer_12/width_16k` JumpReLU SAE
on a 16 GB GPU, applies seven editing operators at a matched forget level, and scores ~2700
generations with two OpenRouter LLM judges (`$1.07` total). **None of that GPU / paid-API work
runs here.** Instead we load the experiment's *recorded per-prompt judge scores* and
**reproduce the downstream statistical analysis** with the original functions:

1. the per-prompt **joint utility** = `harmonic_mean(fluency, content_preservation)`,
2. the **paired bootstrap advantage** of the discovered single-absorber ablation (**KG-ABL**) vs.
   two dense baselines — the strongest *ungated* difference-of-means erasure (**DENSE-SUB-ABL**)
   and the genuinely-fair *gated* control (**DENSE-SUB-ABL-GATED-FAIR**),
3. the per-case **3-way fork** verdict and the **headline tally**.

**Headline result reproduced below:** `DISCOVERY_IS_THE_VALUE_FAIR_GATE_CLOSES_GAP` — KG-ABL beats
the *ungated* dense baseline, but the *genuinely-fair* gated dense control **closes the gap**, so
the SAE's contribution is **label-free WHERE-to-gate discovery**, not edit-quality magic. And
**concentration/precision — not the absorption diagnostic — predicts whether forgetting happens.**


In [ ]:
# --- Install dependencies (Colab-safe) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# This demo only needs numpy + matplotlib, both pre-installed on Colab.
# On Colab: skip (installing would corrupt the pre-loaded C extensions).
# Locally: install at Colab's exact versions so the env matches Colab.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')


In [ ]:
# --- Imports (original script used numpy + math; matplotlib added for the demo viz) ---
import json, math, os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# numpy 2.0 compat shims (harmless no-ops if the attributes already exist)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "product"): np.product = np.prod


In [ ]:
# --- Data loading helper (GitHub URL with local fallback, for Colab compatibility) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-8/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
# --- Load the curated demo data and unpack the three blocks ---
data = load_data()
meta            = data["metadata"]                  # run config + headline summary
edit_per_prompt = data["edit_per_prompt"]           # per (case, role, prompt) LLM-judge scores
per_case        = data["kg_vs_controls_per_case"]   # per-case fork verdicts (all 8 cases)

print("model      :", meta["model"])
print("SAE        :", meta["sae"]["release"], "/", meta["sae"]["sae_params"])
print("gating     : cosine=%.4f  L0=%.1f  pass=%s" % (
    meta["gating_check"]["cosine"], meta["gating_check"]["L0"], meta["gating_check"]["pass"]))
print("operators  :", len(meta["operators"]))
print("judges     : %s + %s  (spent $%.3f over %d calls)" % (
    meta["judge"]["primary_model"], meta["judge"]["second_judge_model"],
    meta["judge"]["spent_usd"], meta["judge"]["n_calls"]))
print("per-prompt judged rows :", len(edit_per_prompt))
print("per-case verdict rows  :", len(per_case))


## Configuration

All tunable knobs live here. The only real "scale" knob is **`B_BOOT`**, the number of paired
bootstrap resamples used for the confidence intervals. The point estimates (`diff`) are exact and
do not depend on it; larger `B_BOOT` only sharpens the CI. We start at a demo minimum and note the
true original value (`10000`).


In [ ]:
# ---- Tunable parameters ----
SEED       = meta["seed"]            # 1234 (from the recorded run)
# B_BOOT controls only the bootstrap-CI sharpness; the point estimates are exact regardless.
# Demo minimum is 2000; the full original (10000) is sub-second here, so we use it.
# B_BOOT   = 2000                    # <- demo minimum
B_BOOT     = meta["B_boot"]          # full original = 10000 (bootstrap is cheap: numpy on ~20-30 elem arrays)
MIN_PAIRED = 6                       # min paired prompts required to compute a bootstrap CI (original gate)
PRES       = ("RETAIN", "UNRELATED") # preservation roles scored for the joint utility (original constant)

# The curated mini-demo carries per-prompt judge scores for these representative cases:
DEMO_CASES = sorted({r["metadata_case"] for r in edit_per_prompt})
print("recomputing per-prompt advantages for:", DEMO_CASES)
print("B_BOOT =", B_BOOT, "(full original =", meta["B_boot"], ")")


## Original analysis functions

These are copied (essentially verbatim) from the experiment's `method.py` / `core.py`:

- `harmonic_mean(f, c)` — the per-prompt **joint utility** combining fluency and content-preservation.
- `paired_bootstrap_diff(a, b)` — the paired-bootstrap difference-in-means with a 95% percentile CI.
- `paired_util(...)` — builds the paired per-prompt joint-utility arrays for two operators over the
  preservation roles. (The original read a nested in-memory `judged`/`gen` structure; here it reads
  the same numbers back from the flat curated rows. Logic unchanged.)

Operator suffixes in the data: `kg` = KG-ABL (discovered single absorber, *ours*),
`sub` = DENSE-SUB-ABL (strongest **ungated** dense, lead comparator),
`fair` = DENSE-SUB-ABL-GATED-FAIR (the **genuinely-fair** gated dense control).


In [ ]:
rng = np.random.default_rng(SEED)

def harmonic_mean(f, c):                         # method.py: verbatim
    f = float(f); c = float(c)
    if f <= 0 and c <= 0:
        return 0.0
    return (2.0 * f * c) / (f + c + 1e-9)

def paired_bootstrap_diff(a, b, B=None):         # core.py: verbatim (B defaults to B_BOOT)
    if B is None:
        B = B_BOOT
    a = np.asarray(a, float); b = np.asarray(b, float)
    n = len(a)
    if n == 0:
        return {"diff": 0.0, "ci_lo": 0.0, "ci_hi": 0.0, "excl_0": False, "n": 0}
    idx = rng.integers(0, n, size=(B, n))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": float(a.mean() - b.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "excl_0": bool(lo > 0 or hi < 0), "n": int(n)}

# _paired_util adapted to read the flat curated rows; joint utility = harmonic_mean(fluency, content_pres)
def paired_util(rows, opA, opB, roles=PRES):
    a, b = [], []
    for r in rows:
        if r["metadata_role"] not in roles:
            continue
        fa, ca = r.get("metadata_fluency_%s" % opA), r.get("metadata_content_pres_%s" % opA)
        fb, cb = r.get("metadata_fluency_%s" % opB), r.get("metadata_content_pres_%s" % opB)
        if None in (fa, ca, fb, cb):
            continue
        a.append(harmonic_mean(fa, ca))
        b.append(harmonic_mean(fb, cb))
    return np.array(a), np.array(b)

def _favors_kg(ci):                              # method.py: verbatim
    return bool(ci is not None and ci.get("excl_0") and ci.get("diff", 0) > 0)

print("functions ready (rng seeded with %d)" % SEED)


## Step 1 — Recompute the KG-vs-control advantages from per-prompt judge scores

For each demo case we build the paired per-prompt joint utilities over the preservation roles and
bootstrap the difference:

- **`adv_KGvsSUB`** = advantage of KG-ABL over the strongest **ungated** dense erasure, and
- **`adv_KGvsFAIR`** = advantage of KG-ABL over the **genuinely-fair** gated dense control.

A positive value means KG-ABL preserves unrelated/retained behaviour *better* than the baseline at
the matched forget level. We print our recomputed value next to the value recorded in the run — they
match exactly (the bootstrap CI may differ in the last digits because of RNG state).


In [ ]:
pub = {c["metadata_case"]: c for c in per_case}
recomputed = {}

hdr = "%-20s | %-26s | %-26s" % ("case", "adv_KGvsSUB (recomp / pub)", "adv_KGvsFAIR (recomp / pub)")
print(hdr); print("-" * len(hdr))
for case in DEMO_CASES:
    rows  = [r for r in edit_per_prompt if r["metadata_case"] == case]
    uK_s, uS = paired_util(rows, "kg", "sub")
    uK_f, uF = paired_util(rows, "kg", "fair")
    ci_sub  = paired_bootstrap_diff(uK_s, uS) if len(uK_s) >= MIN_PAIRED else None
    ci_fair = paired_bootstrap_diff(uK_f, uF) if len(uK_f) >= MIN_PAIRED else None
    adv_sub  = ci_sub["diff"]  if ci_sub  else 0.0
    adv_fair = ci_fair["diff"] if ci_fair else 0.0
    recomputed[case] = {"ci_sub": ci_sub, "ci_fair": ci_fair, "adv_sub": adv_sub, "adv_fair": adv_fair}
    p = pub[case]
    print("%-20s | %+9.4f  /  %+9.4f  excl0=%-5s | %+9.4f  /  %+9.4f  excl0=%s" % (
        case, adv_sub, p["metadata_adv_KGvsSUB"], ci_sub["excl_0"] if ci_sub else None,
        adv_fair, p["metadata_adv_KGvsFAIR"], ci_fair["excl_0"] if ci_fair else None))
print("\nKG beats the UNGATED dense (adv_KGvsSUB > 0, CI excludes 0) but the advantage over the")
print("genuinely-FAIR gated dense collapses to ~0 (adv_KGvsFAIR CI includes 0): the fair gate closes the gap.")


## Step 2 — The 3-way fork, per case

The experiment's per-case verdict follows a simple fork (reproduced from `method.py`):

```
if not kg_can_forget:           NO_MEANINGFUL_FORGET   (single-latent ablation cannot induce real forgetting)
elif sub_favors and fair_favors: KG_BEATS_STRONGEST_AND_FAIR_GATED
else:                            FAIR_GATED_CLOSES_GAP  (discovery is the value)
```

`sub_favors` / `fair_favors` come from our recomputed CIs (`_favors_kg`). `kg_can_forget` is a
**model-internal** signal (completion-accuracy drop + frozen sub-probe drop) measured on the GPU at
run time, so we read it from the recorded per-case record. We confirm our recomputed fork matches the
recorded `fork_verdict`.


In [ ]:
F_BEATS  = "KG_BEATS_STRONGEST_AND_FAIR_GATED"
F_CLOSES = "FAIR_GATED_CLOSES_GAP_DISCOVERY_IS_THE_VALUE"
F_NOMEAN = "NO_MEANINGFUL_FORGET_SCOPE_TO_PARTIAL_SUPPRESSION"
SHORT = {F_BEATS: "KG_BEATS", F_CLOSES: "FAIR_CLOSES_GAP", F_NOMEAN: "NO_MEANINGFUL_FORGET"}

print("%-20s %-10s %-11s %-12s %-18s %s" % (
    "case", "kg_forget", "sub_favors", "fair_favors", "fork (recomputed)", "matches?"))
print("-" * 86)
for case in DEMO_CASES:
    rc = recomputed[case]
    kg_can_forget = bool(pub[case]["metadata_kg_can_forget"])
    sub_favors  = _favors_kg(rc["ci_sub"])
    fair_favors = _favors_kg(rc["ci_fair"])
    if not kg_can_forget:
        fork = F_NOMEAN
    elif sub_favors and fair_favors:
        fork = F_BEATS
    else:
        fork = F_CLOSES
    match = "OK" if fork == pub[case]["metadata_fork_verdict"] else "DIFF"
    print("%-20s %-10s %-11s %-12s %-18s %s" % (
        case, kg_can_forget, sub_favors, fair_favors, SHORT[fork], match))


## Step 3 — Headline tally across all 8 cases

The full run covered 8 cases (4 concentrated absorbers + 4 references). We only carry per-prompt
scores for 3 in this mini demo, so the tally below uses the recorded `fork_verdict` of every case.


In [ ]:
tally = Counter(SHORT.get(c["metadata_fork_verdict"], c["metadata_fork_verdict"]) for c in per_case)
print("3-way fork tally over all %d cases:\n" % len(per_case))
for k in ["KG_BEATS", "FAIR_CLOSES_GAP", "NO_MEANINGFUL_FORGET"]:
    members = [c["metadata_case"] for c in per_case if SHORT.get(c["metadata_fork_verdict"]) == k]
    print("  %-22s %d   %s" % (k, tally.get(k, 0), ", ".join(members)))
print("\nn_concentrated_wins (KG beats a fair gate on a concentrated absorber):",
      meta["summary"]["n_concentrated_wins"])
print("OVERALL VERDICT:", meta["summary"]["overall_verdict"])


## Step 4 — Concentration, not the absorption diagnostic, predicts forgetting

Distributed country senses (Jordan, US — and Cook) do **not** meaningfully forget despite clean
firing signatures, while a concentrated co-firing latent (insult) does. So the win predictor is the
**lexical concentration / sub-context precision** of the target sense, not whether absorption was
diagnosed.


In [ ]:
print("%-20s %-13s %-11s %-11s %s" % ("case", "concentration", "regime", "kg_forget", "fork"))
print("-" * 70)
for c in per_case:
    print("%-20s %-13s %-11s %-11s %s" % (
        c["metadata_case"], c["metadata_concentration"], c["metadata_regime"],
        c["metadata_kg_can_forget"], SHORT.get(c["metadata_fork_verdict"])))
print("\nnote:", meta["summary"]["concentration_attribution"]["note"])


## Visualization

**Left:** per-case joint-utility advantage of KG-ABL over the *ungated* dense (blue) and over the
*genuinely-fair* gated dense (orange). Blue bars are clearly positive (KG beats the ungated baseline)
while orange bars sit at ~0 (the fair gate closes the gap). **Right:** the 3-way fork tally.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cases_order = [c["metadata_case"] for c in per_case]
advSUB  = [c["metadata_adv_KGvsSUB"]  for c in per_case]
advFAIR = [c["metadata_adv_KGvsFAIR"] for c in per_case]
x = np.arange(len(cases_order)); w = 0.38

ax = axes[0]
ax.bar(x - w/2, advSUB,  w, label="KG vs strongest UNGATED dense", color="#2c7fb8")
ax.bar(x + w/2, advFAIR, w, label="KG vs genuinely-FAIR gated dense", color="#d95f0e")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", "\n") for c in cases_order], fontsize=7)
ax.set_ylabel("joint-utility advantage (KG minus control)")
ax.set_title("KG beats the UNGATED dense — but the FAIR gate closes the gap")
ax.legend(fontsize=8)

ax = axes[1]
labels = ["KG_BEATS", "FAIR_CLOSES_GAP", "NO_MEANINGFUL_FORGET"]
counts = [tally.get(l, 0) for l in labels]
ax.bar(range(len(labels)), counts, color=["#999999", "#d95f0e", "#bbbbbb"])
for i, v in enumerate(counts):
    ax.text(i, v + 0.04, str(v), ha="center", fontsize=11)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8, rotation=12)
ax.set_ylabel("number of cases")
ax.set_ylim(0, max(counts) + 0.6)
ax.set_title("3-way fork tally (8 cases)")

plt.tight_layout()
plt.show()


## Honest negatives (verbatim from the run)

The experiment ships 30 verbatim honest-negative notes; the mini demo carries the first few. They
record the de-inflation (the iter-7 footprint-gated headline over-erased ~14x), the fair-gate-closes
finding, and that gating is prior art — i.e. the SAE's value is the *label-free discovery of where to
gate*, not a magic edit operator.


In [ ]:
for i, h in enumerate(meta["honest_negatives"], 1):
    print("%d. %s\n" % (i, h))
